# Lecture Notes: The Magic Ship Problem (Binary Search on Answer)

This lecture discusses a complex search problem called **Magic Ship**, originally from Codeforces Round 60. It serves as a prime example of how to apply **Binary Search** to find an optimal "minimum number of days" in a movement-constrained environment.

---

## 1. Problem Overview
The problem is modeled as a navigation task on a 2D Cartesian plane.

### The Setup
- **Initial Position**: $(x1, y1)$
- **Destination**: $(x2, y2)$
- **Wind Behavior**: A periodic weather forecast of length $n$ is provided. Each day, the wind pushes the ship one unit in a specific direction: North (**U**), South (**D**), Left (**L**), or Right (**R**).
- **Captain’s Control**: Every day, the captain can choose to move the ship one unit in any of the four directions (U, D, L, R) or stay put.
- **Combined Movement**: The ship's final position each day is the vector sum of the wind's movement and the captain's movement.


### The Goal
Find the **minimum number of days** required to reach the destination. If it is impossible, return -1.

---

## 2. Key Insights and Logic
The lecture identifies several critical properties that allow for an efficient solution.

### Periodicity of Weather
The weather forecast string repeats indefinitely. If the forecast is "UR" ($n=2$), on day 1 the wind moves Up, day 2 Right, day 3 Up, and so on.

### Separating Forces
A crucial realization is that the movements are **additive and commutative**. To check if a destination is reachable in $k$ days, we can:
1. Calculate where the wind alone would take the ship after $k$ days (let’s call this point $(x3, y3)$).
2. Determine if the captain can use their $k$ "steering units" to cover the remaining distance to $(x2, y2)$.

### Reachability Criteria: Manhattan Distance
The captain can reach the destination in $k$ days if and only if the **Manhattan Distance** between the wind-drifted position $(x3, y3)$ and the target $(x2, y2)$ is less than or equal to $k$.
- **Manhattan Distance Formula**: $|x2 - x3| + |y2 - y3|$
- **Logic**: Since the captain can move one unit per day, in $k$ days, they can move a total distance of $k$ units across both axes.

---

## 3. Algorithm: Binary Search on the Answer
The problem asks for a minimum value (days). The lecture demonstrates that if a destination is reachable in $k$ days, it is also reachable in $k+1$ days (the captain can simply "nullify" the wind or stay put). This **monotonicity** allows for Binary Search.

### Determining the Search Space
- **Lower Bound**: 0 days.
- **Upper Bound**: The lecture provides a proof that if a solution exists, it will be found within $D \times n$ days, where $D$ is the initial distance and $n$ is the forecast length. For implementation, a large constant like $10^{14}$ or $10^{15}$ is sufficient.

### Pseudocode for Feasibility Check
This routine determines if $k$ days are sufficient.

```python
FUNCTION Is_Feasible(k, n, start_pos, target_pos, forecast_offsets):
    # 1. Calculate full cycles and remainder
    num_cycles = k / n
    remainder_days = k % n
    
    # 2. Calculate wind's total drift (x3, y3)
    # Use precomputed offsets for one full cycle and remainder
    wind_x = (num_cycles * total_cycle_x_offset) + forecast_offsets[remainder_days].x
    wind_y = (num_cycles * total_cycle_y_offset) + forecast_offsets[remainder_days].y
    
    current_x = start_pos.x + wind_x
    current_y = start_pos.y + wind_y
    
    # 3. Calculate distance to target
    distance_needed = absolute_value(target_pos.x - current_x) + 
                       absolute_value(target_pos.y - current_y)
    
    # 4. Return true if captain can cover the distance
    IF distance_needed <= k:
        RETURN True
    ELSE:
        RETURN False
```

### Main Binary Search Pseudocode
```python
INITIALIZE low = 0
INITIALIZE high = 10^15
INITIALIZE answer = -1

WHILE low <= high:
    mid = (low + high) / 2
    
    IF Is_Feasible(mid):
        answer = mid      # This might be the minimum
        high = mid - 1    # Try to find a smaller number of days
    ELSE:
        low = mid + 1     # Need more days to reach target

RETURN answer
```

---

## 4. Impossible Scenarios
The lecture clarifies that some cases are impossible (e.g., wind always blowing North while the destination is South). 
- If the binary search finishes and no $k$ was ever found feasible, the output is **-1**.
- This is proven because the captain must be able to reduce the distance by at least 1 unit every cycle to eventually reach the goal.

---

## 5. Implementation Details: Prefix Sums
To avoid calculating the wind's drift from scratch for every "mid" value:
- **Precompute a "Walk List"**: Store the cumulative $(x, y)$ displacement for each day of the first weather cycle.
- This allows $O(1)$ calculation of the wind's position for any $k$ by using `k / n` (full cycles) and `k % n` (lookup in the list).

---

## Summary of Takeaways
- **Monotonicity**: The core reason Binary Search works here is that if $k$ days work, any $k' > k$ also works.
- **Coordinate Independence**: Treating the wind drift and captain's movement as separate components simplifies the logic.
- **Manhattan Constraint**: The captain's total movement budget is exactly $k$ units, which must cover the gap created by the wind.
- **Upper Bound Intuition**: Choosing a sufficiently large upper bound ($10^{15}$) ensures that all possible valid paths are covered within the search space.
